#Использование нейросети для датирования исторических текстов на нидерландском языке (XIII-XIX вв.)

Помимо классических моделей, в рамках проекта было решено попробовать использовать нейросети для датирования текстов на нидерландском языке.  

Механизм работы нейросетевого классификатора заключается в следующем:
* текст, разбитый на предложения, направляется в Bert;
* для токенов (слов и частей слов) с помощью Bert  извлекаются эмбеддинги;
* далее полученные эмбеддинги направляются в класификационный слой для отнесения предложения к тому или иному временному периоду.

Для обучения были использованы (для сравнения) три предобученные модели Bert:
1) [GysBert](https://huggingface.co/emanjavacas/GysBERT) -  модель, которая обучалась частично на том же корпусе, что используется в проекте (корпус DBNL);
2) [Language Model for Historic Dutch](https://huggingface.co/dbmdz/bert-base-historic-dutch-cased) - модель, которая обучалась на другом корпусе исторических текстов (Delpher Corpus, включает отцифрованные тексты газет на нидерландском языке, датируемые 1618-1879 гг.).
3) [Bertje](https://huggingface.co/GroNLP/bert-base-dutch-cased) - модель, которая обучалась на современных текстах на нидерландском языке.

##1. Загрузка библиотек

In [ ]:
#импорт библиотек для работы с torch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW

In [ ]:
#для работы с датафреймами
import pandas as pd

#для работы с регулярными выражениями
import re

#для работы с массивами
import numpy as np

In [ ]:
#для отслеживания хода выполнения кода
from tqdm import tqdm
tqdm.pandas()

In [ ]:
#для разделения на train, test, подсчета метрик, кодировки таргета, подсчета весов классов
#StratifiedGroupKFold - для того, чтобы предложения из одного и того же текста не попали одновременно в train и test и не образовали утечку данных
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

##2. Деление текстов на предложения и разбивка на train и test

In [ ]:
df_selected_bert = pd.read_csv('/content/df_merged_CLEANED.csv')

In [ ]:
for_column_year = [
    (df_selected_bert['_jaar'] < 1500, '1200-1500 гг.'),
    (df_selected_bert['_jaar'] < 1700, '1500-1700 гг.'),
    (df_selected_bert['_jaar'] < 1900, '1700-1900 гг.'),
]

In [ ]:
df_selected_bert['eeuw'] = df_selected_bert['_jaar'].case_when(caselist=for_column_year)

*Функция разбивки на предложения для работы с GysBert (uncased)*

In [ ]:
def sent(text):
  try:
    clean_text = text.split('onderscheiden van de rest van de tekst door middel van accolades')[1]
  except:
    clean_text = text
  clean_text = re.sub(r'{[=\w\s><\.:-]+}', ' ', clean_text)
  clean_text = re.sub(r'\d', ' ', clean_text)
  clean_text = re.sub(r'[\!\?]', '.', clean_text)
  clean_text = re.sub(r'[,\\\/""''><;{}\n\t&%$#@*”“‘’()§=:]+', ' ', clean_text)
  clean_text = re.sub(r'\xa0', ' ', clean_text)
  clean_text = re.sub(r'[\[\]]', '', clean_text)
  clean_text = re.sub(r'[^\w\s\.]', ' ', clean_text)
  clean_text = re.sub(r'\b[A-Za-z]\b', ' ', clean_text)
  clean_text = re.sub(r' +', ' ', clean_text)
  clean_text = clean_text.lower()

  sentences = clean_text.split('.')

  return sentences

*Функция разбивки на предложения для работы с Language Model for Historic Dutch и Bertje (cased)*

In [ ]:
def sent(text):
  try:
    clean_text = text.split('onderscheiden van de rest van de tekst door middel van accolades')[1]
  except:
    clean_text = text
  clean_text = re.sub(r'{[=\w\s><\.:-]+}', ' ', clean_text)
  clean_text = re.sub(r'\d', ' ', clean_text)
  clean_text = re.sub(r'[\!\?]', '.', clean_text)
  clean_text = re.sub(r'[,\\\/""''><;{}\n\t&%$#@*”“‘’()§=:]+', ' ', clean_text)
  clean_text = re.sub(r'\xa0', ' ', clean_text)
  clean_text = re.sub(r'[\[\]]', '', clean_text)
  clean_text = re.sub(r'[^\w\s\.]', ' ', clean_text)
  clean_text = re.sub(r'\b[A-Za-z]\b', ' ', clean_text)
  clean_text = re.sub(r' +', ' ', clean_text)
  # clean_text = clean_text.lower()

  sentences = clean_text.split('.')

  return sentences

In [ ]:
for_bert_clean = df_selected_bert[['ti_id', 'text', 'eeuw']].copy()

In [ ]:
for_bert_clean['sent'] = for_bert_clean['text'].progress_apply(sent)

100%|██████████| 833/833 [01:21<00:00, 10.18it/s]


In [ ]:
exploded_for_bert = for_bert_clean.explode('sent')

*Отбор предложений от 5 до 128 токенов*

In [ ]:
def number_tokens(sentence):
  tokens = sentence.split()
  num_tokens = len(tokens)
  return num_tokens

In [ ]:
exploded_for_bert['num_tokens'] = exploded_for_bert['sent'].progress_apply(number_tokens)

100%|██████████| 4616084/4616084 [00:08<00:00, 547946.50it/s]


In [ ]:
exploded_for_bert_short = exploded_for_bert[exploded_for_bert['num_tokens'] < 128]

In [ ]:
exploded_for_bert_short = exploded_for_bert_short[exploded_for_bert_short['num_tokens'] > 5]

*Разбивка на train и test в отношении 80 % / 20 %*

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42) #чтобы предложения из одного текста не попали одновременно в train и test

In [ ]:
for train_idx, test_idx in sgkf.split(exploded_for_bert_short['sent'], exploded_for_bert_short['eeuw'], groups=exploded_for_bert_short['ti_id']):
    train_df = exploded_for_bert_short.iloc[train_idx]
    test_df = exploded_for_bert_short.iloc[test_idx]
    break

In [ ]:
print("Соотношение классов в train и test")
print("Train:\n", train_df['eeuw'].value_counts(normalize = True))
print("Test:\n", test_df['eeuw'].value_counts(normalize = True))

Train:
* 1700-1900 гг. - 0.812489
* 1500-1700 гг. - 0.154729
* 1200-1500 гг. - 0.032782

Test:
* 1700-1900 гг. - 0.806855
* 1500-1700 гг. - 0.142965
* 1200-1500 гг. - 0.050180

*Формирование случайной выборки из предложений (для уменьшения времени работы модели из-за ограничений использования GPU в Google Colab)*

In [ ]:
sample_sizes_train = {'1200-1500 гг.': 30000, '1500-1700 гг.': 50000, '1700-1900 гг.': 70000}

In [ ]:
sample_sizes_test = {'1200-1500 гг.': 7000, '1500-1700 гг.': 10000, '1700-1900 гг.': 13000}

In [ ]:
df_sampled_train = train_df.groupby('eeuw', group_keys=False).apply(
    lambda x: x.sample(n=sample_sizes_train.get(x.name, 1), random_state=42)
)

/tmp/ipykernel_1472/3271174012.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sampled_train = train_df.groupby('eeuw', group_keys=False).apply(


In [ ]:
df_sampled_test = test_df.groupby('eeuw', group_keys=False).apply(
    lambda x: x.sample(n=sample_sizes_test.get(x.name, 1), random_state=42)
)

/tmp/ipykernel_1472/1258950599.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sampled_test = test_df.groupby('eeuw', group_keys=False).apply(


##3. Модели

! Код написан с использованием нейросетевых помощников, а именно Gemini.

Был задан промпт, в котором содержалась просьба к Gemini дать пример кода для нейросети, в рамках работы которой из текстов извлекались бы эмбеддинги с помощью Bert, а затем эти эмбеддинги передавались бы в классификационный слой.

###3.1. Определение классов и функций

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #перевод на использование GPU, если он доступен

In [ ]:
#определение класса для обработки текста
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = [str(t) for t in texts] #для перестраховки - конвертация в строковый формат
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): #функция подсчета длины текста
        return len(self.texts)

    def __getitem__(self, idx): #функция возвращает закодированный с текст с паддингом до максимальной длины 128, механизм внимания и лейблы, всё в формате тензоров
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [ ]:
#определение класса-классификатора - GysBERTClassifier
class GysBERTClassifier(nn.Module):
    def __init__(self, model_name_or_path, num_classes=3): #значение классов - 3, по количеству временных периодов
        super(GysBERTClassifier, self).__init__() #отсылка к материнскому классу - nn.Module
        self.bert = AutoModel.from_pretrained(model_name_or_path) #загрузка предобученной модели
        self.dropout = nn.Dropout(0.1) #отключение части нейронов для стабилизации модели
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes) #классификационный слой

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask) #получение эмбеддингов
        token_embeddings = outputs.last_hidden_state #берем последний слой

        mask_expanded = attention_mask.unsqueeze(-1)
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask #получение эмбеддингов не для каждого конкретного токена, а для предложения в целом

        x = self.dropout(mean_pooled)
        logits = self.classifier(x) #получение предсказания
        return logits

In [ ]:
#определение класса-классификатора - Language Model for Historic Dutch (аналогично)
#на самом деле, смысла задавать отдельные классы для разных моделей Bert в моем проекте нет, т.к. они делают одно и то же
#но я оставила это в коде, так как добавляла разные модели постепенно и не хотела в них запутаться (сначала думала попробовать только GysBert, но в итоге запустила 3 разные модели Bert)
class dbmdzBERTClassifier(nn.Module):
    def __init__(self, model_name_or_path, num_classes=3):
        super(dbmdzBERTClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name_or_path)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state

        mask_expanded = attention_mask.unsqueeze(-1)
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        x = self.dropout(mean_pooled)
        logits = self.classifier(x)
        return logits

In [ ]:
#определение класса-классификатора - BERTJEClassifier (аналогично)
#см. примечание в предыдущей ячейке относительно того, почему оставила код с одим и тем же по фукнционалу классом, но с разными названиями
class BERTJEClassifier(nn.Module):
    def __init__(self, model_name_or_path, num_classes=3):
        super(BERTJEClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name_or_path)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state

        mask_expanded = attention_mask.unsqueeze(-1)
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        x = self.dropout(mean_pooled)
        logits = self.classifier(x)
        return logits

In [ ]:
#функция обучения
def train_epoch(model, data_loader, optimizer, loss_fn, device):
    model.train() #переводим модель в состояние обучения
    total_loss = 0

    for batch in data_loader:
        optimizer.zero_grad() #перед началом новой эпохи сбрасываем градиенты с предыдущей эпохи

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask) #делаем предсказание с помощью заданного классифкатора
        loss = loss_fn(logits, labels) #подсчет loss, то есть отклонения предсказания (класса с наибольшей вероятностью) от действительного значения
        total_loss += loss.item()

        loss.backward() #обратный проход для определения того, насколько нужно скорректировать модель, чтобы предсказания были точнее
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) #нейронка предложила этот кусочек кода, чтобы не было "градиентного взрыва", т.е. когда градиент превышает 1.0 и оптимайзер делает слишком большой шаг
        optimizer.step() #шаг оптимайзера для корректировки работы модели по итогам подсчета loss

    return total_loss / len(data_loader)

In [ ]:
#функция оценки
def validate_epoch(model, data_loader, loss_fn, device):
    model.eval() #перевод модели в режим валидации
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad(): #отключаем подсчет градиентов, т.к. он нужен был в обучении, а на этапе валидации градиенты не обновляются
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask) #предсказание
            loss = loss_fn(logits, labels) #подсчет loss
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1) #"сырые" предсказания (то есть вероятности классов) переводятся в предсказание конкретного класса (то есть класса, которому присвоена наибольшая вероятность)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = total_loss / len(data_loader)
    val_acc = accuracy_score(all_labels, all_preds) #подсчет метрик
    val_f1_score = f1_score(all_labels, all_preds, average='weighted')

    return val_loss, val_acc, val_f1_score, all_labels, all_preds

###3.2. Обучение и метрики

####3.2.1. Gysbert

In [ ]:
%%time
if __name__ == "__main__":
    #train
    train_texts = df_sampled_train['sent'].tolist() #предложения
    train_labels = df_sampled_train['eeuw'].tolist() #лейблы

    #test
    val_texts = df_sampled_test['sent'].tolist()
    val_labels = df_sampled_test['eeuw'].tolist()

    #кодирование лейблов
    label_encoder = LabelEncoder()
    numerical_train_labels = label_encoder.fit_transform(train_labels)
    numerical_val_labels = label_encoder.transform(val_labels)

    #подсчет весов классов для их балансировки
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(numerical_train_labels),
        y=numerical_train_labels)

    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

    print(f"Calculated imbalance compensation weights: {class_weights}")

    #модель
    model_path = "emanjavacas/GysBERT"

    #токенайзер
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    #упаковка предложений и лейблов в dataloader
    train_dataset = TextClassificationDataset(train_texts, numerical_train_labels, tokenizer)
    val_dataset = TextClassificationDataset(val_texts, numerical_val_labels, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    #определение модели и оптимайзера
    model = GysBERTClassifier(model_name_or_path=model_path, num_classes=3).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    #определение loss-функции, включение в нее параметра ранее подсчитанного веса классов
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

    # количество эпох (выбрано так мало эпох из-за ограничений Google Colab по использованию GPU +
    #по итогам работы моделей, когда выяснилось, что для двух моделей уже после первой эпохи loss на валидации начинает повышаться, то есть модель переобучается)
    epochs = 2

    #путь для сохранение модели
    save_path = "/content/final_gysbert_classifier.pt"

    best_val_loss = float('inf')
    best_true_lbl = []
    best_pred_lbl = []

    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch + 1}/{epochs} ---")

        #Обучение
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        print(f"Training Loss: {train_loss:.4f}")

        #Валидация
        val_loss, val_acc, val_f1_score, true_lbl, pred_lbl = validate_epoch(model, val_loader, loss_fn, device)
        print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f} | Validation F1-score: {val_f1_score:.4f}")

        #Сохранение лучшей модели по значению loss на валидации
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_true_lbl = true_lbl
            best_pred_lbl = pred_lbl

            torch.save(model.state_dict(), save_path)
            print(f"💾 New best model found! Saved to '{save_path}'")

    #метрики
    print("\n========== Evaluation Report (Best Model) ==========")
    print(classification_report(best_true_lbl, best_pred_lbl, digits=4, zero_division=0))

Calculated imbalance compensation weights: [1.66666667 1.         0.71428571]


config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/441M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emanjavacas/GysBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Epoch 1/2 ---


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Training Loss: 0.1529
Validation Loss: 0.2530 | Validation Accuracy: 0.9343 | Validation F1-score: 0.9345
💾 New best model found! Saved to '/content/final_gysbert_classifier.pt'

--- Epoch 2/2 ---
Training Loss: 0.1025
Validation Loss: 0.3707 | Validation Accuracy: 0.9332 | Validation F1-score: 0.9333

========== Evaluation Report (Best Model) ==========
              precision    recall  f1-score   support

           0     0.9667    0.8910    0.9273      7000
           1     0.8869    0.9438    0.9144     10000
           2     0.9572    0.9503    0.9538     13000

    accuracy                         0.9343     30000
   macro avg     0.9369    0.9284    0.9318     30000
weighted avg     0.9360    0.9343    0.9345     30000

CPU times: user 2h 2s, sys: 7.99 s, total: 2h 10s
Wall time: 2h 1min 6s


In [ ]:
# cl_r = classification_report(best_true_lbl, best_pred_lbl, output_dict=True, target_names=label_encoder.classes_)
# df_cl_r = pd.DataFrame(cl_r).T
# df_cl_r.to_excel('GysBert_model.xlsx')

####3.2.2. Language Model for Historic Dutch

In [ ]:
#то же самое, но для другой модели Берт
%%time
if __name__ == "__main__":

    train_texts = df_sampled_train['sent'].tolist()
    train_labels = df_sampled_train['eeuw'].tolist()

    val_texts = df_sampled_test['sent'].tolist()
    val_labels = df_sampled_test['eeuw'].tolist()

    label_encoder = LabelEncoder()
    numerical_train_labels = label_encoder.fit_transform(train_labels)
    numerical_val_labels = label_encoder.transform(val_labels)

    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(numerical_train_labels),
        y=numerical_train_labels)

    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

    print(f"Calculated imbalance compensation weights: {class_weights}")

    model_path = "dbmdz/bert-base-historic-dutch-cased"

    tokenizer = AutoTokenizer.from_pretrained(model_path)

    train_dataset = TextClassificationDataset(train_texts, numerical_train_labels, tokenizer)
    val_dataset = TextClassificationDataset(val_texts, numerical_val_labels, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    model = dbmdzBERTClassifier(model_name_or_path=model_path, num_classes=3).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

    epochs = 2

    save_path = "/content/final_dbmdz_classifier.pt"

    best_val_loss = float('inf')
    best_true_lbl = []
    best_pred_lbl = []

    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch + 1}/{epochs} ---")

        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        print(f"Training Loss: {train_loss:.4f}")

        val_loss, val_acc, val_f1_score, true_lbl, pred_lbl = validate_epoch(model, val_loader, loss_fn, device)
        print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f} | Validation F1-score: {val_f1_score:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_true_lbl = true_lbl
            best_pred_lbl = pred_lbl

            torch.save(model.state_dict(), save_path)
            print(f"💾 New best model found! Saved to '{save_path}'")

    print("\n========== Evaluation Report (Best Model) ==========")
    print(classification_report(best_true_lbl, best_pred_lbl, digits=4, zero_division=0))


Calculated imbalance compensation weights: [1.66666667 1.         0.71428571]


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/220k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-historic-dutch-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Epoch 1/2 ---
Training Loss: 0.1714
Validation Loss: 0.2117 | Validation Accuracy: 0.9455 | Validation F1-score: 0.9458
💾 New best model found! Saved to '/content/final_dbmdz_classifier.pt'

--- Epoch 2/2 ---
Training Loss: 0.1057
Validation Loss: 0.2234 | Validation Accuracy: 0.9432 | Validation F1-score: 0.9435

========== Evaluation Report (Best Model) ==========
              precision    recall  f1-score   support

           0     0.9766    0.9250    0.9501      7000
           1     0.9033    0.9468    0.9245     10000
           2     0.9639    0.9556    0.9597     13000

    accuracy                         0.9455     30000
   macro avg     0.9479    0.9425    0.9448     30000
weighted avg     0.9467    0.9455    0.9458     30000

CPU times: user 2h 7min 22s, sys: 12.3 s, total: 2h 7min 34s
Wall time: 2h 8min 57s


In [ ]:
# cl_r = classification_report(best_true_lbl, best_pred_lbl, output_dict=True, target_names=label_encoder.classes_)
# df_cl_r = pd.DataFrame(cl_r).T
# df_cl_r.to_excel('dbmdz_model.xlsx')

####3.2.3. Bertje

In [ ]:
#то же самое, но для ещё одной модели Берт
%%time
if __name__ == "__main__":

    train_texts = df_sampled_train['sent'].tolist()
    train_labels = df_sampled_train['eeuw'].tolist()

    val_texts = df_sampled_test['sent'].tolist()
    val_labels = df_sampled_test['eeuw'].tolist()

    label_encoder = LabelEncoder()
    numerical_train_labels = label_encoder.fit_transform(train_labels)
    numerical_val_labels = label_encoder.transform(val_labels)

    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(numerical_train_labels),
        y=numerical_train_labels)

    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

    print(f"Calculated imbalance compensation weights: {class_weights}")

    model_path = "GroNLP/bert-base-dutch-cased"
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    train_dataset = TextClassificationDataset(train_texts, numerical_train_labels, tokenizer)
    val_dataset = TextClassificationDataset(val_texts, numerical_val_labels, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    model = BERTJEClassifier(model_name_or_path=model_path, num_classes=3).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

    epochs = 2

    save_path = "/content/final_Bertje_classifier.pt"

    best_val_loss = float('inf')
    best_true_lbl = []
    best_pred_lbl = []

    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch + 1}/{epochs} ---")

        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        print(f"Training Loss: {train_loss:.4f}")

        val_loss, val_acc, val_f1_score, true_lbl, pred_lbl = validate_epoch(model, val_loader, loss_fn, device)
        print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f} | Validation F1-score: {val_f1_score:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_true_lbl = true_lbl
            best_pred_lbl = pred_lbl

            torch.save(model.state_dict(), save_path)
            print(f"💾 New best model found! Saved to '{save_path}'")

    print("\n========== Evaluation Report (Best Model) ==========")
    print(classification_report(best_true_lbl, best_pred_lbl, digits=4, zero_division=0))


Calculated imbalance compensation weights: [1.66666667 1.         0.71428571]


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: GroNLP/bert-base-dutch-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Epoch 1/2 ---
Training Loss: 0.1865
Validation Loss: 0.3004 | Validation Accuracy: 0.9242 | Validation F1-score: 0.9246
💾 New best model found! Saved to '/content/final_Bertje_classifier.pt'

--- Epoch 2/2 ---
Training Loss: 0.1071
Validation Loss: 0.2921 | Validation Accuracy: 0.9322 | Validation F1-score: 0.9323
💾 New best model found! Saved to '/content/final_Bertje_classifier.pt'

========== Evaluation Report (Best Model) ==========
              precision    recall  f1-score   support

           0     0.9403    0.9426    0.9414      7000
           1     0.8980    0.9172    0.9075     10000
           2     0.9550    0.9381    0.9465     13000

    accuracy                         0.9322     30000
   macro avg     0.9311    0.9326    0.9318     30000
weighted avg     0.9326    0.9322    0.9323     30000

CPU times: user 2h 1min 37s, sys: 10.9 s, total: 2h 1min 48s
Wall time: 2h 2min 59s


In [ ]:
# cl_r = classification_report(best_true_lbl, best_pred_lbl, output_dict=True, target_names=label_encoder.classes_)
# df_cl_r = pd.DataFrame(cl_r).T
# df_cl_r.to_excel('Bertje_model.xlsx')

*Звуковой сигнал, чтобы не пропустить окончание работы модели*

In [ ]:
from google.colab import output
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

####3.2.4. Выводы
1) Несмотря на то, что Language Model for Historic Dutch предобучалась на  корпусе исторических текстов, датируемых 1618-1879 гг., то есть не охватывала тексты с XIII по XVI вв., которые есть в моем корпусе, она всё равно хорошо справилась с задачей датировки текстов, в том числе первого класса, даже лучше, чем GysBert.

Как предположение, это может быть связано, в частности, с тем, что Language Model for Historic Dutch учитывает регистр, в отличие от GysBert, поэтому лучше улавливает нюансы текстов.

2) Bertje, который предобучался на современных текстах, может потребоваться больше эпох для достижения лучших результатов, чем историческим моделям: на второй эпохе loss на валидации у Bert снижается, в отличие от исторических моделей, то есть модель совершенствуется.

Но, конечно, для всех моделей хорошо было бы попробовать больше эпох, как минимум пять, чтобы посмотреть на динамику loss на валидации, поэтому этот вывод лишь предварительный.

3) Как и в классических моделях, в метриках немного "проседает" второй класс.
Это может свидетельствовать о том, что избранное деление на классы можно совершенствовать, сдвигая границы классов / выделяя новые классы.